In [ ]:
# 1. Mount Google Drive.
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
# 2. Configure the model pairs and freshly extract the exact codebase archive.
import shutil
from datetime import datetime
from pathlib import Path
from zipfile import ZipFile
from zoneinfo import ZoneInfo

DRIVE_DIR = Path("/content/drive/MyDrive/[ICLR] Embedding KD")
ARCHIVE_PATH = DRIVE_DIR / "ICLR-MDD-nqd_claude_rcm.zip"
EXTRACT_DIR = Path("/content/ICLR-MDD-nqd_claude_rcm_workspace")
# Add or remove entries here. Slugs must be unique because they name every cache,
# checkpoint, log and result table belonging to one teacher/student pair.
MODEL_PAIRS = [
    {
        "slug": "qwen3_0_6b_to_minilmv2_l6_h384",
        "teacher": "Qwen/Qwen3-Embedding-0.6B",
        "student": "nreimers/MiniLMv2-L6-H384-distilled-from-BERT-Large",
        "teacher_pooling": "last_token",
        "student_special_token": "[CLS]",
        "teacher_special_token": "G",
        "batch_size": 32,
        "learning_rate": 2e-5,
    },
    {
        "slug": "bge_m3_to_minilmv2_l6_h768",
        "teacher": "BAAI/bge-m3",
        "student": "nreimers/MiniLMv2-L6-H768-distilled-from-BERT-Large",
        "teacher_pooling": "cls",
        "student_special_token": "[CLS]",
        "teacher_special_token": "<s>",
        "batch_size": 32,
        "learning_rate": 2e-5,
    },
    {
        "slug": "qwen3_4b_to_bert_base",
        "teacher": "Qwen/Qwen3-Embedding-4B",
        "student": "google-bert/bert-base-uncased",
        "teacher_pooling": "last_token",
        "student_special_token": "##",
        "teacher_special_token": "G",
        "batch_size": 32,
        "learning_rate": 2e-5,
    },
]
EPOCHS = 5

# One timestamp per execution of this cell, reused by every Drive output folder so a
# new run never overwrites the log, tables or weights of a previous one.
RUN_STAMP = datetime.now(ZoneInfo("Asia/Ho_Chi_Minh")).strftime("%Y%m%d-%H%M%S")
EXPERIMENT_NAME = f"model_pair_comparison_{RUN_STAMP}"

assert DRIVE_DIR.is_dir(), f"Google Drive directory not found: {DRIVE_DIR}"
assert ARCHIVE_PATH.is_file(), f"Codebase archive not found: {ARCHIVE_PATH}"
assert EXTRACT_DIR == Path("/content/ICLR-MDD-nqd_claude_rcm_workspace")

# Always start from the code in the selected ZIP, never a stale extracted repo.
if EXTRACT_DIR.is_symlink():
    EXTRACT_DIR.unlink()
elif EXTRACT_DIR.exists():
    shutil.rmtree(EXTRACT_DIR)
EXTRACT_DIR.mkdir(parents=True, exist_ok=False)

extract_root = EXTRACT_DIR.resolve()
with ZipFile(ARCHIVE_PATH) as archive:
    unsafe_members = []
    for member in archive.infolist():
        destination = (EXTRACT_DIR / member.filename).resolve()
        if destination != extract_root and extract_root not in destination.parents:
            unsafe_members.append(member.filename)
    assert not unsafe_members, f"Unsafe paths in ZIP: {unsafe_members[:5]}"
    archive.extractall(EXTRACT_DIR)

# Accept a ZIP with or without one enclosing top-level folder. Training is driven by
# importing the codebase, so the markers are the modules this notebook imports.
repo_candidates = sorted(
    {
        main_path.parent.resolve()
        for main_path in EXTRACT_DIR.rglob("main.py")
        if (main_path.parent / "distiller.py").is_file()
        and (main_path.parent / "config" / "talas_config.py").is_file()
    },
    key=lambda path: (len(path.parts), str(path)),
)
assert repo_candidates, (
    "The ZIP was extracted, but no TALAS repo containing main.py, distiller.py "
    "and config/talas_config.py was found."
)
PROJECT_DIR = repo_candidates[0]
if len(repo_candidates) > 1:
    print(f"Multiple repo roots found; using the shallowest: {PROJECT_DIR}")

# Common paths and one isolated run context per model pair.
TRAIN_DATA = PROJECT_DIR / "data" / "train_set" / "merged_3_data_5k_each.csv"
TALAS_CACHE_DIR = PROJECT_DIR / "cache" / "talas"
DRIVE_EXPERIMENT_DIR = DRIVE_DIR / "runs" / EXPERIMENT_NAME
COMPARISON_BY_EPOCH_CSV = DRIVE_EXPERIMENT_DIR / "comparison_by_epoch.csv"
COMPARISON_FINAL_CSV = DRIVE_EXPERIMENT_DIR / "comparison_final.csv"
RUNS = []
for pair in MODEL_PAIRS:
    run_dir = PROJECT_DIR / "models" / "talas" / pair["slug"]
    drive_run_dir = DRIVE_EXPERIMENT_DIR / pair["slug"]
    RUNS.append(
        {
            **pair,
            "run_dir": run_dir,
            "metrics_path": run_dir / "metrics.jsonl",
            "step_metrics_path": run_dir / "step_metrics.jsonl",
            "teacher_cache_path": TALAS_CACHE_DIR / f"{pair['slug']}_teacher_train.pt",
            "drive_run_dir": drive_run_dir,
            "log_path": drive_run_dir / "train.log",
            "test_by_epoch_csv": drive_run_dir / "test_by_epoch.csv",
            "final_test_csv": drive_run_dir / "final_test_results.csv",
            "plots_dir": drive_run_dir / "plots",
            "train_by_step_csv": drive_run_dir / "train_by_step.csv",
            "train_by_epoch_csv": drive_run_dir / "train_by_epoch.csv",
            "weights_dir": drive_run_dir / "weights",
        }
    )

assert (PROJECT_DIR / "requirements.txt").is_file(), "requirements.txt is missing"
assert TRAIN_DATA.is_file(), f"Training data not found: {TRAIN_DATA}"
# Evaluation reads the test split only, so no val_set file is required here.
required_split_files = {
    "train_set": {
        "merged_3_data_5k_each.csv", "banking77_train.csv",
        "emotion_train.csv", "tweet_train.csv",
    },
    "test_set": {
        "banking77_test.csv", "emotion_test.csv", "tweet_test.csv",
        "mrpc_test.csv", "scitail_test.csv", "wic_test.csv",
        "sick_test.csv", "sts12_test.csv", "stsb_test.csv",
    },
}
for split_dir, filenames in required_split_files.items():
    split_path = PROJECT_DIR / "data" / split_dir
    assert split_path.is_dir(), f"Data split directory not found: {split_path}"
    missing = [name for name in filenames if not (split_path / name).is_file()]
    assert not missing, f"Missing files in {split_path}: {missing}"
slugs = [run["slug"] for run in RUNS]
assert slugs and len(slugs) == len(set(slugs)), f"Model-pair slugs must be unique: {slugs}"
# Refuse to reuse a timestamped experiment instead of overwriting a previous run.
DRIVE_EXPERIMENT_DIR.mkdir(parents=True, exist_ok=False)
for run in RUNS:
    run["drive_run_dir"].mkdir(parents=True, exist_ok=False)
    run["weights_dir"].mkdir(parents=True, exist_ok=False)

print("Resolved Colab paths:")
for name, path in {
    "archive": ARCHIVE_PATH,
    "extract": EXTRACT_DIR,
    "project": PROJECT_DIR,
    "train_data": TRAIN_DATA,
    "drive_output": DRIVE_EXPERIMENT_DIR,
}.items():
    print(f"  {name:12s}: {path}")
print(f"  {'experiment':12s}: {EXPERIMENT_NAME}")
print("Model pairs:")
for index, run in enumerate(RUNS, start=1):
    print(f"  {index}. {run['slug']}: {run['teacher']} -> {run['student']}")

In [ ]:
# 3. Run from the repo root so the relative paths inside the codebase resolve, and
# make the repo importable: training runs inside this kernel, not in a subprocess.
import sys

%cd $PROJECT_DIR

if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))
print(f"sys.path[0]: {sys.path[0]}")

In [ ]:
# 4. Install the project requirements into this Colab kernel.
# %pip streams its output straight into this cell.
%pip install -r requirements.txt

In [ ]:
# 5. Fail early if the extracted codebase or any model pair is incompatible.
from config.talas_config import TALASConfig
from transformers import AutoConfig

cfg = TALASConfig()
assert cfg.distill_method == "talas", cfg.distill_method
assert cfg.eval_every == 1, f"Expected eval_every=1, got {cfg.eval_every}"
assert cfg.train_data_path == "data/train_set/merged_3_data_5k_each.csv"

for run in RUNS:
    student_hf_config = AutoConfig.from_pretrained(run["student"])
    teacher_hf_config = AutoConfig.from_pretrained(
        run["teacher"], trust_remote_code=True
    )
    student_hidden = getattr(student_hf_config, "hidden_size", None)
    teacher_hidden = getattr(teacher_hf_config, "hidden_size", None)
    assert student_hidden, f"No hidden_size in student config: {run['student']}"
    assert teacher_hidden, f"No hidden_size in teacher config: {run['teacher']}"
    student_layers = getattr(student_hf_config, "num_hidden_layers", "?")
    run["student_hidden_size"] = student_hidden
    run["teacher_hidden_size"] = teacher_hidden
    print(
        f"[READY] {run['slug']}: student L{student_layers}/H{student_hidden}, "
        f"teacher H{teacher_hidden}, pooling={run['teacher_pooling']}"
    )
print("[READY] TALAS configuration is ready.")
print(
    "If pip asked for a restart, use Runtime > Restart session and re-run from cell 3."
)

In [ ]:
# 6. Report the accelerator that KnowledgeDistiller will use in this kernel.
import torch

cuda_count = torch.cuda.device_count()
mps_available = hasattr(torch.backends, "mps") and torch.backends.mps.is_available()
if cuda_count >= 2:
    selected = "student=cuda:0, teacher=cuda:1"
elif torch.cuda.is_available():
    selected = "student=cuda:0, teacher=cuda:0"
elif mps_available:
    selected = "student=mps, teacher=mps"
else:
    selected = "student=cpu, teacher=cpu"

print(f"Python environment: {sys.executable}")
print(f"PyTorch version: {torch.__version__} (CUDA build {torch.version.cuda})")
print(f"CUDA available: {torch.cuda.is_available()} (device count={cuda_count})")
for index in range(cuda_count):
    properties = torch.cuda.get_device_properties(index)
    print(
        f"  cuda:{index}: {properties.name} "
        f"({properties.total_memory / (1024 ** 3):.1f} GiB)"
    )
print(f"MPS available: {mps_available}")
print(f"KnowledgeDistiller will use: {selected}")
if torch.cuda.is_available():
    print("GPU STATUS: READY - TALAS training will use CUDA.")
else:
    print("GPU STATUS: NOT USING CUDA - enable a Colab GPU runtime before training.")

In [ ]:
# 7. Remove only caches and local outputs belonging to the configured pairs.
import shutil

for run in RUNS:
    for path in (
        run["teacher_cache_path"],
        run["run_dir"],
    ):
        resolved = path.resolve()
        assert PROJECT_DIR.resolve() in resolved.parents, f"Unsafe cleanup target: {resolved}"
        if path.is_symlink():
            path.unlink()
            print(f"Removed stale symlink: {path}")
        elif path.is_dir():
            shutil.rmtree(resolved)
            print(f"Removed stale artifacts: {resolved}")
        elif path.exists():
            path.unlink()
            print(f"Removed stale artifact: {resolved}")
    run["run_dir"].mkdir(parents=True, exist_ok=False)
    print(f"Clean run output: {run['run_dir']}")

In [ ]:
# 8. Train every model pair sequentially. Each pair gets an isolated Drive log,
# cache, checkpoint directory and weights directory. Evaluation reads test_set only.
import gc
import io
import os
from contextlib import redirect_stderr, redirect_stdout

import distiller as distiller_module
from distiller import KnowledgeDistiller
from src.evaluation.evaluation_automodel import (
    test_cls_tasks,
    test_pair_tasks,
    test_sts_tasks,
)

# KnowledgeDistiller.evaluate() resolves its per-epoch task lists from these module
# globals, so rebinding them points the per-epoch pass at the test files. Pair
# thresholds are refit on the test pairs instead of being carried over from val_set.
distiller_module.eval_cls_tasks = test_cls_tasks
distiller_module.eval_pair_tasks = test_pair_tasks
distiller_module.eval_sts_tasks = test_sts_tasks
for task_paths in (test_cls_tasks, test_pair_tasks, test_sts_tasks):
    for entry in task_paths:
        for path in (entry if isinstance(entry, tuple) else (entry,)):
            assert "val_set" not in path, f"Evaluation must not read val_set: {path}"

# The training corpus is drawn from EMOTION, WiC and STS-B. The remaining six
# benchmarks measure out-of-distribution transfer. Every benchmark contributes one
# primary score on a 0-1 scale: macro-F1 (classification), AP (pair), or Spearman.
IOD_BENCHMARKS = frozenset({"emotion", "wic", "stsb"})
PRIMARY_METRICS = {
    "classification": "f1",
    "pair": "average_precision",
    "sts": "spearman",
}


def benchmark_name(path):
    name = Path(path).stem
    return name[:-len("_test")] if name.endswith("_test") else name


def summarize_evaluation(payload):
    scores = {}
    for family, metric in PRIMARY_METRICS.items():
        for path, result in payload.get(family, {}).items():
            value = result if family == "sts" else result[metric]
            scores[benchmark_name(path)] = float(value)
    iod = [score for name, score in scores.items() if name in IOD_BENCHMARKS]
    ood = [score for name, score in scores.items() if name not in IOD_BENCHMARKS]
    assert iod and ood, f"Expected both IOD and OOD benchmarks, got {sorted(scores)}"
    return {
        "avg_iod": sum(iod) / len(iod),
        "avg_ood": sum(ood) / len(ood),
        "avg_all": sum(scores.values()) / len(scores),
    }


class TeeStream(io.TextIOBase):
    """Write to the notebook cell and to the Drive log, collapsing tqdm redraws."""

    def __init__(self, stream, handle):
        self.stream = stream
        self.handle = handle
        self._pending = ""

    def write(self, text):
        self.stream.write(text)
        self.stream.flush()
        self._pending += text
        wrote_line = False
        while "\n" in self._pending:
            line, self._pending = self._pending.split("\n", 1)
            # An in-place progress bar only needs its last redraw in the log file.
            self.handle.write(line.rsplit("\r", 1)[-1] + "\n")
            wrote_line = True
        self._pending = self._pending.rsplit("\r", 1)[-1]
        if wrote_line:
            self.handle.flush()
        return len(text)

    def flush(self):
        self.stream.flush()
        self.handle.flush()

    def writable(self):
        return True

    def isatty(self):
        return False


class TestSplitDistiller(KnowledgeDistiller):
    """Evaluate test_set and log only IOD, OOD and overall averages."""

    def print_evaluation_table(self, split, results):
        summary = summarize_evaluation(results)
        title = (
            f"TEST - EPOCH {self.current_epoch + 1}"
            if split != "test"
            else "FINAL TEST"
        )
        print(f"\n{title}")
        print(f"  IOD avg : {100.0 * summary['avg_iod']:.2f}")
        print(f"  OOD avg : {100.0 * summary['avg_ood']:.2f}")
        print(f"  avg     : {100.0 * summary['avg_all']:.2f}\n")
        return summary

os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"
os.environ["WANDB_MODE"] = "disabled"

cell_stdout = sys.stdout
for pair_index, run in enumerate(RUNS, start=1):
    config = TALASConfig()
    config.train_data_path = str(TRAIN_DATA)
    config.student_model_name = run["student"]
    config.teacher_model_name = run["teacher"]
    config.student_special_token = run["student_special_token"]
    config.teacher_special_token = run["teacher_special_token"]
    config.pooling_method = run["teacher_pooling"]
    config.cache_path = str(run["teacher_cache_path"])
    config.batch_size = run["batch_size"]
    config.epochs = EPOCHS
    config.learning_rate = run["learning_rate"]
    config.max_length = 256
    config.save_dir = str(run["run_dir"])
    config.weights_dir = str(run["weights_dir"])
    config.eval_every = 1
    config.use_wandb = False
    run["config"] = config

    print("\n" + "#" * 80)
    print(f"MODEL PAIR {pair_index}/{len(RUNS)}: {run['slug']}")
    print("#" * 80)
    print(f"Student model : {config.student_model_name}")
    print(f"Teacher model : {config.teacher_model_name}")
    print(f"Teacher pool  : {config.pooling_method}")
    print(f"Batch size    : {config.batch_size}")
    print(f"Epochs        : {config.epochs}")
    print(f"Learning rate : {config.learning_rate}")
    print(f"Evaluation    : IOD avg / OOD avg / avg on data/test_set/*")
    print(f"Output        : {run['run_dir']}")
    print(f"Drive log     : {run['log_path']}")
    print(f"Drive weights : {run['weights_dir']}")
    print("#" * 80)

    with run["log_path"].open("w", encoding="utf-8") as log_handle:
        tee = TeeStream(cell_stdout, log_handle)
        # stderr is merged so tqdm bars land in both the cell and the pair log.
        with redirect_stdout(tee), redirect_stderr(tee):
            print("=" * 70)
            print(f"Configuration for {config.distill_method.upper()} method:")
            print("=" * 70)
            for key, value in config.to_dict().items():
                print(f"  {key:25s} : {value}")
            print("=" * 70 + "\n")
            distiller = TestSplitDistiller(config)
            distiller.train()
        tee.flush()
    # Detach the closed file before the next loop iteration releases TeeStream.
    tee.handle = io.StringIO()

    assert run["metrics_path"].is_file(), (
        f"Training finished without metrics: {run['metrics_path']}"
    )
    expected_weight_files = [
        run["weights_dir"] / f"student_epoch_{epoch}.pt"
        for epoch in range(1, EPOCHS + 1)
    ]
    missing_weights = [
        path for path in expected_weight_files
        if not path.is_file() or path.stat().st_size == 0
    ]
    assert not missing_weights, f"Missing or empty weights: {missing_weights}"
    print(f"[OK] Finished {run['slug']}; log: {run['log_path']}")

    # The next pair must not inherit model objects or reserved GPU memory.
    del distiller
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print(f"\n[OK] Finished all {len(RUNS)} model pairs")

In [ ]:
# 9. Build one summary row per pair/epoch and one final row per pair.
# No classification/pair/STS family means are logged or exported.
import json

import pandas as pd
from IPython.display import display


def summary_row(run, payload, epoch=None):
    summary = payload.get("summary") or summarize_evaluation(payload)
    row = {
        "pair": run["slug"],
        "teacher": run["teacher"],
        "student": run["student"],
        "IOD avg": float(summary["avg_iod"]),
        "OOD avg": float(summary["avg_ood"]),
        "avg": float(summary["avg_all"]),
    }
    if epoch is not None:
        row["epoch"] = int(epoch)
    return row


epoch_rows = []
final_rows = []
for run in RUNS:
    with run["metrics_path"].open(encoding="utf-8") as handle:
        records = [json.loads(line) for line in handle if line.strip()]

    pair_epoch_rows = []
    for record in records:
        payload = record.get("validation")
        epoch = record.get("train", {}).get("epoch")
        if payload and epoch is not None:
            pair_epoch_rows.append(summary_row(run, payload, epoch))
    assert pair_epoch_rows, (
        f"No per-epoch evaluation records found in {run['metrics_path']}"
    )

    test_payloads = [record["test"] for record in records if record.get("test")]
    assert len(test_payloads) == 1, (
        f"Expected one final test record for {run['slug']}, got {len(test_payloads)}"
    )
    pair_final_row = summary_row(run, test_payloads[0])
    pd.DataFrame(pair_epoch_rows).to_csv(run["test_by_epoch_csv"], index=False)
    pd.DataFrame([pair_final_row]).to_csv(run["final_test_csv"], index=False)
    epoch_rows.extend(pair_epoch_rows)
    final_rows.append(pair_final_row)

test_by_epoch = pd.DataFrame(epoch_rows)[
    ["pair", "teacher", "student", "epoch", "IOD avg", "OOD avg", "avg"]
].sort_values(["pair", "epoch"]).reset_index(drop=True)
final_test_results = pd.DataFrame(final_rows)[
    ["pair", "teacher", "student", "IOD avg", "OOD avg", "avg"]
].sort_values("pair").reset_index(drop=True)
test_by_epoch.to_csv(COMPARISON_BY_EPOCH_CSV, index=False)
final_test_results.to_csv(COMPARISON_FINAL_CSV, index=False)
print("TEST SUMMARY BY EPOCH")
display(test_by_epoch.style.format(precision=4))
print("FINAL MODEL-PAIR COMPARISON")
display(final_test_results.style.format(precision=4))
print(f"Saved: {COMPARISON_BY_EPOCH_CSV}")
print(f"Saved: {COMPARISON_FINAL_CSV}")

In [ ]:
# 10. Set up plotting and a loader for each model pair's metrics.
# metrics.jsonl holds one record per epoch; step_metrics.jsonl holds one per optimizer
# step. Epoch means cannot show *when* inside an epoch a curve flattened, so the step
# file drives every training figure and the epoch file only supplies the per-epoch
# markers.
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

COMPARISON_PLOTS_DIR = DRIVE_EXPERIMENT_DIR / "plots"
COMPARISON_PLOTS_DIR.mkdir(parents=True, exist_ok=True)


def activate_plot_run(plot_run):
    """Load one pair's metrics into the globals used by the plot helpers."""
    global PLOT_RUN_SLUG, PLOTS_DIR, cfg, config, test_by_epoch_plot
    global steps, epochs

    PLOT_RUN_SLUG = plot_run["slug"]
    metrics_path = plot_run["metrics_path"]
    step_metrics_path = plot_run["step_metrics_path"]
    PLOTS_DIR = plot_run["plots_dir"]
    cfg = plot_run["config"]
    config = cfg
    test_by_epoch_plot = test_by_epoch[
        test_by_epoch["pair"] == PLOT_RUN_SLUG
    ].copy()
    PLOTS_DIR.mkdir(parents=True, exist_ok=True)

    assert step_metrics_path.is_file(), (
        f"No per-step metrics at {step_metrics_path}. This file is written by "
        "KnowledgeDistiller.log_step_records(); an older codebase ZIP will not have it."
    )
    with step_metrics_path.open(encoding="utf-8") as handle:
        steps = pd.DataFrame([json.loads(line) for line in handle if line.strip()])
    steps = steps.sort_values("global_step").reset_index(drop=True)

    with metrics_path.open(encoding="utf-8") as handle:
        records = [json.loads(line) for line in handle if line.strip()]
    epochs = pd.DataFrame(
        [record["train"] for record in records if record.get("train")]
    ).sort_values("epoch").reset_index(drop=True)

    steps.to_csv(plot_run["train_by_step_csv"], index=False)
    epochs.to_csv(plot_run["train_by_epoch_csv"], index=False)
    print(f"\nDetailed plots for: {PLOT_RUN_SLUG}")
    print(f"Steps logged : {len(steps)} across {steps['epoch'].nunique()} epochs")
    print(f"Step columns : {', '.join(sorted(steps.columns))}")
    print(f"Saved {plot_run['train_by_step_csv']}")
    print(f"Saved {plot_run['train_by_epoch_csv']}")

# ---- palette -------------------------------------------------------------------
# Categorical slots in fixed order -- a series keeps its hue no matter how many other
# series are on the axes, so the same colour never means two different things across
# figures. Slots 3 and 4 sit below 3:1 against the surface, so every line also carries
# a direct label at its right end and the CSVs above are the table view.
SURFACE = "#fcfcfb"
INK = "#0b0b0b"
INK_MUTED = "#898781"
GRID = "#e1e0d9"
AXIS = "#c3c2b7"
SERIES = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300"]
POS, NEG = "#2a78d6", "#d03b3b"  # diverging pair for signed deltas

# seaborn's whitegrid supplies the base; the rc block below replaces every colour it
# would otherwise pick, so the theme is this palette and not seaborn's defaults.
sns.set_theme(
    context="notebook",
    style="whitegrid",
    palette=sns.color_palette(SERIES),
    rc={
        "figure.facecolor": SURFACE,
        "axes.facecolor": SURFACE,
        "savefig.facecolor": SURFACE,
        "axes.edgecolor": AXIS,
        "axes.labelcolor": INK,
        "axes.titlecolor": INK,
        "axes.titlesize": 11,
        "axes.titleweight": "bold",
        "axes.titlelocation": "left",
        # Room between the title and the axes for the legend, which is parked there
        # rather than inside the plot: every series also carries a label at its right
        # end, and an in-plot legend collided with them.
        "axes.titlepad": 22,
        "axes.labelsize": 9,
        "axes.grid": True,
        "axes.axisbelow": True,
        "grid.color": GRID,
        "grid.linewidth": 0.8,
        "grid.linestyle": "-",
        "xtick.color": INK_MUTED,
        "ytick.color": INK_MUTED,
        "xtick.labelsize": 8,
        "ytick.labelsize": 8,
        "text.color": INK,
        "legend.frameon": False,
        "legend.fontsize": 8,
        "lines.linewidth": 2.0,
        "figure.dpi": 130,
        "savefig.dpi": 200,
        "savefig.bbox": "tight",
    },
)


def ema(values, span):
    """Exponential moving average. Raw per-step curves are too noisy to read a trend
    off; the raw trace is kept underneath at low alpha so the noise stays visible."""
    return pd.Series(values, dtype="float64").ewm(span=max(2, span), adjust=False).mean()


def style_axes(ax):
    sns.despine(ax=ax)
    ax.grid(axis="x", visible=False)
    return ax


def mark_epochs(ax, frame=None):
    """Vertical rule at each epoch boundary -- the candidate sets are redrawn there, so
    a discontinuity across one is resampling, not a training instability."""
    frame = steps if frame is None else frame
    for boundary in frame.groupby("epoch")["global_step"].min().iloc[1:]:
        ax.axvline(boundary, color=AXIS, linewidth=0.8, linestyle=(0, (4, 3)), zorder=0)


def label_ends(ax, placements):
    """Direct labels at the right end of each line, nudged apart when they collide.

    Slots 3 and 4 are below 3:1 on this surface, so identity may not rest on the hue
    alone; these labels (plus the legend) are that relief.
    """
    span = ax.get_ylim()[1] - ax.get_ylim()[0]
    minimum_gap = span * 0.045
    placements = sorted(placements, key=lambda item: item[1])
    for index in range(1, len(placements)):
        x, y, text, colour = placements[index]
        previous_y = placements[index - 1][1]
        if y - previous_y < minimum_gap:
            placements[index] = (x, previous_y + minimum_gap, text, colour)
    left, right = ax.get_xlim()
    x_span = right - left
    widest = max(len(text) for _, _, text, _ in placements)
    gutter = x_span * min(0.42, 0.016 * widest + 0.04)
    for x, y, text, colour in placements:
        ax.annotate(
            text, (x, y), xytext=(x + x_span * 0.025, y), textcoords="data",
            color=colour, fontsize=8, va="center", ha="left", clip_on=False,
        )
    # The gutter is label space, not data space: leaving ticks in it would advertise a
    # range the run never reached.
    data_right = max(x for x, _, _, _ in placements)
    ax.set_xlim(left, right + gutter)
    ax.set_xticks([tick for tick in ax.get_xticks() if left <= tick <= data_right])
    ax.set_xlim(left, right + gutter)


def place_legend(ax, count, below=False):
    """Legend outside the plot area. Inside it collided with the end labels.

    Above the axes by default; `below` is for the panels whose entries wrap to two
    rows, where the top slot runs into the title.
    """
    if count <= 1:
        if ax.get_legend():
            ax.get_legend().remove()
        return
    if below:
        sns.move_legend(ax, loc="upper center", bbox_to_anchor=(0.5, -0.17),
                        ncol=min(count, 2), title=None, borderaxespad=0.0,
                        columnspacing=1.6)
    else:
        sns.move_legend(ax, loc="lower center", bbox_to_anchor=(0.5, 1.0),
                        ncol=min(count, 3), title=None, borderaxespad=0.0,
                        columnspacing=1.6)


def step_lines(ax, columns, labels, smooth=None, show_raw=True):
    """Plot per-step series with EMA on top of the raw trace, legend and end labels.

    Returns the columns actually drawn, so a caller can tell whether a figure had
    anything to show: a metric a method does not emit is simply skipped.
    """
    available = [column for column in columns if column in steps.columns]
    if not available:
        return []
    smooth = len(steps) // 40 if smooth is None else smooth
    placements = []
    for column in available:
        slot = columns.index(column)
        colour = SERIES[slot % len(SERIES)]
        label = labels[slot]
        values = steps[column].to_numpy(dtype="float64")
        if show_raw:
            sns.lineplot(x=steps["global_step"], y=values, ax=ax, color=colour,
                         linewidth=0.7, alpha=0.18, zorder=1, errorbar=None)
        smoothed = ema(values, smooth)
        sns.lineplot(x=steps["global_step"], y=smoothed, ax=ax, color=colour,
                     label=label, zorder=2, errorbar=None)
        placements.append((steps["global_step"].iloc[-1], smoothed.iloc[-1], label, colour))
    mark_epochs(ax)
    style_axes(ax)
    ax.set_xlabel("optimizer step")
    # A lone series needs neither a legend nor an end label -- the title names it, and
    # both would just repeat the title while eating plot width.
    place_legend(ax, len(available))
    if len(available) > 1:
        label_ends(ax, placements)
    return available


def finish(fig, path, note=None, note_y=-0.02):
    # note_y is lowered on figures whose legends sit below the axes, so the caption
    # clears them.
    if note:
        fig.text(0, note_y, note, color=INK_MUTED, fontsize=8, ha="left", va="top")
    fig.savefig(path)
    plt.close(fig)
    print(f"Saved {path.name}")



In [ ]:
# 11. Render detailed figures for every model pair plus one all-pair comparison.
from IPython.display import Image, display


def render_current_plot_run():
    # ---- 1. train loss ---------------------------------------------------------------
    fig, ax = plt.subplots(figsize=(9, 3.6))
    step_lines(ax, ["loss"], ["training loss"])
    ax.set_ylabel("loss (nats)")
    ax.set_title("Training loss per step")
    finish(fig, PLOTS_DIR / "01_train_loss.png",
           "Faint trace: raw per-step loss. Solid: EMA. Dashed rules: epoch boundaries, "
           "where the candidate sets are redrawn.")
    
    # ---- 7. IOD/OOD/overall test evaluation per epoch -------------------------------
    summary_columns = ["IOD avg", "OOD avg", "avg"]
    summary_colours = [SERIES[0], SERIES[1], INK]
    fig, ax = plt.subplots(figsize=(9, 3.8))
    placements = []
    for column, colour in zip(summary_columns, summary_colours):
        sns.lineplot(data=test_by_epoch_plot, x="epoch", y=column, label=column,
                     color=colour, marker="o", markersize=4.5, ax=ax, errorbar=None)
        last = test_by_epoch_plot.sort_values("epoch").iloc[-1]
        placements.append((last["epoch"], last[column], column, colour))
    style_axes(ax)
    ax.set_title(f"Test summary per epoch — {PLOT_RUN_SLUG}")
    ax.set_xlabel("epoch")
    ax.set_ylabel("score")
    ax.set_xticks(sorted(test_by_epoch_plot["epoch"].unique()))
    place_legend(ax, len(summary_columns))
    label_ends(ax, placements)
    finish(fig, PLOTS_DIR / "07_test_summary_by_epoch.png",
           "IOD: emotion, WiC, STS-B. OOD: the six held-out benchmarks. Overall is "
           "the unweighted mean of all nine benchmark primary scores.")
    
    # ---- 9. the training system itself -----------------------------------------------
    fig, axes = plt.subplots(1, 2, figsize=(9, 3.2))
    step_lines(axes[0], ["lr_next"], ["learning rate"], show_raw=False)
    axes[0].set_title("Learning rate schedule")
    axes[0].set_ylabel("lr")
    axes[0].ticklabel_format(axis="y", style="sci", scilimits=(0, 0))
    step_lines(axes[1], ["step_seconds"], ["seconds per step"])
    axes[1].set_title("Step time")
    axes[1].set_ylabel("seconds")
    peak = epochs["peak_memory_mb"].max() if "peak_memory_mb" in epochs.columns else None
    finish(fig, PLOTS_DIR / "09_training_system.png",
           f"Peak allocated GPU memory across the run: {peak:.0f} MB." if peak else None)
    
    print(f"\nDetailed figures in Drive: {PLOTS_DIR}")
    for path in sorted(PLOTS_DIR.glob("*.png")):
        display(Image(filename=str(path)))


for plot_run in RUNS:
    activate_plot_run(plot_run)
    render_current_plot_run()


# Render the experiment-level comparison once, after every pair-specific plot set.
summary_columns = ["IOD avg", "OOD avg", "avg"]
summary_colours = [SERIES[0], SERIES[1], INK]
# ---- 8. final comparison across teacher/student pairs ----------------------------
comparison_long = final_test_results.melt(
    id_vars=["pair"], value_vars=summary_columns,
    var_name="summary", value_name="score",
)
fig, ax = plt.subplots(figsize=(9, max(3.2, 0.8 * len(RUNS) + 1.8)))
sns.barplot(data=comparison_long, x="score", y="pair", hue="summary",
            hue_order=summary_columns, palette=summary_colours, orient="h", ax=ax)
for container in ax.containers:
    ax.bar_label(container, fmt="%.4f", padding=3, fontsize=8)
style_axes(ax)
ax.grid(axis="y", visible=False)
ax.grid(axis="x", visible=True)
ax.set_xlabel("final test score")
ax.set_ylabel("")
ax.set_title("Final teacher/student pair comparison")
ax.set_xlim(0, min(1.0, comparison_long["score"].max() * 1.18))
place_legend(ax, len(summary_columns))
finish(fig, COMPARISON_PLOTS_DIR / "final_model_pair_comparison.png",
       "Primary score per benchmark: classification macro-F1, pair AP, and STS "
       "Spearman; group values are unweighted benchmark means.")

print(f"Comparison figures in Drive: {COMPARISON_PLOTS_DIR}")
for path in sorted(COMPARISON_PLOTS_DIR.glob("*.png")):
    display(Image(filename=str(path)))
